# Monsoon Onset 2025 — IMD 1° Rainfall  (Colab Demo)
### ICTS Summer School on Monsoon Dynamics & Climate, 2026

Detect and visualise the gridded monsoon onset date for **2025** using the IMD 1-degree daily rainfall dataset.

**What we do:**
1. 🔧 Install dependencies and clone the analysis package from GitHub
2. ☁️ Mount Google Drive to access the IMD rainfall and threshold data
3. 📦 Imports and config setup
4. 🌡️ Load and visualise the wet-spell rainfall threshold
5. 🌧️ Load IMD rainfall data for 2025
6. 📅 Detect monsoon onset dates (wet-spell / dry-spell veto algorithm)
7. 🗺️ Plot onset date map (monthly stratified colormap)
8. 🔷 Overlay CMZ polygon and extract CMZ onset statistics
9. 📈 Grid-point daily rainfall time-series with onset marker
10. ⏱️ First wet spell vs onset — delay due to dry-spell veto

> **Data you need on Google Drive** (upload once to a folder called `ICTS_monsoon_data`):
> - `imd/data_2025.nc` — IMD 1° daily rainfall 2025
> - `mwset1x1.nc4` — multi-week rainfall threshold climatology
>
> The India shapefile is **bundled in the GitHub repo** — no extra upload needed.


## 🔧 Step 1 — Install Dependencies & Clone the Package

Run this cell **once** at the start of every Colab session.


In [ ]:
import os, subprocess, sys

# ── 1a. Install Python packages not pre-installed on Colab ───────────────────
print("Installing Python packages …")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "gcsfs"], check=True)
print("Python packages ✓")

# ── 1b. Clone the analysis repo (skip if already present) ────────────────────
REPO_URL = "https://github.com/rmasiwal/icts_mcdm_2026.git"
REPO_DIR = "/content/icts_mcdm_2026"

if not os.path.exists(REPO_DIR):
    print("Cloning repo …")
    subprocess.run(["git", "clone", "--depth=1", REPO_URL, REPO_DIR], check=True)
    print("Repo cloned ✓")
else:
    print("Repo already present — pulling latest …")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

# ── 1c. Add repo to Python path ───────────────────────────────────────────────
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f"Repo on path: {REPO_DIR} ✓")


## ☁️ Step 2 — Mount Google Drive & Set Data Paths

Upload the two required files to a folder called **`ICTS_monsoon_data`** in the root of your Google Drive:

| File | What it is |
|------|-----------|
| `imd/data_2025.nc` | IMD 1° daily rainfall 2025 |
| `mwset1x1.nc4` | Multi-week rainfall threshold climatology |

The India shapefile is **bundled in the repo** — no extra upload needed.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# ── Point this to the folder where you uploaded the data ─────────────────────
DRIVE_DATA = "/content/drive/MyDrive/ICTS_monsoon_data"
YEAR       = 2025

IMD_FILE    = f"{DRIVE_DATA}/imd/data_{YEAR}.nc"
THRESH_FILE = f"{DRIVE_DATA}/mwset1x1.nc4"

for label, path in [("IMD rainfall", IMD_FILE),
                    ("Threshold",    THRESH_FILE)]:
    status = "✓  found" if os.path.exists(path) else "✗  MISSING — check the path above"
    print(f"{label:20s}: {status}")


## 📦 Step 3 — Imports & Config

Import all libraries and load the `imd_1deg` config, overriding data paths to point at Google Drive.


In [ ]:
import yaml, importlib
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.gridspec as mgridspec
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.path import Path
from matplotlib.patches import Polygon as MplPolygon

REPO_DIR = "/content/icts_mcdm_2026"

import monsoon_onset.plotter  as _plt
import monsoon_onset.loader   as _ldr
import monsoon_onset.detector as _det
importlib.reload(_plt); importlib.reload(_ldr); importlib.reload(_det)

from monsoon_onset.loader   import load_rainfall
from monsoon_onset.detector import detect_onset, detect_first_wet_spell
from monsoon_onset.plotter  import get_india_outline, doy_to_date_string

%matplotlib inline
os.makedirs(f"{REPO_DIR}/output", exist_ok=True)

# ── Load config, override paths → Google Drive ────────────────────────────────
with open(f"{REPO_DIR}/configs/imd_1deg.yaml") as f:
    cfg = yaml.safe_load(f)

dataset_cfg = cfg["dataset"]
onset_cfg   = cfg["onset"]

dataset_cfg["data_folder"]    = f"{DRIVE_DATA}/imd"
dataset_cfg["threshold_file"] = THRESH_FILE
dataset_cfg["shapefile"]      = f"{REPO_DIR}/data/shapefiles/india_shapefile.shp"
shp_path = dataset_cfg["shapefile"]

print("Config loaded ✓")
print(f"  IMD folder  : {dataset_cfg['data_folder']}")
print(f"  Threshold   : {dataset_cfg['threshold_file']}")
print(f"  Shapefile   : {shp_path}")


## 🌡️ Step 4 — Load & Visualise the Wet-spell Rainfall Threshold

Open `mwset1x1.nc4` and map the `MWmean` variable — the minimum 5-day rainfall required to declare onset at each grid point.


In [ ]:
threshold_var = dataset_cfg.get("threshold_var", "MWmean")
thresh_ds = xr.open_dataset(dataset_cfg["threshold_file"])
thresh_da = thresh_ds[threshold_var]

print(f"Threshold variable : {threshold_var}")
print(f"Threshold shape    : {thresh_da.shape}")
thresh_da


In [ ]:
thresh_plot = thresh_da.squeeze()
boundaries  = np.arange(0, 85, 5)
cmap_obj    = plt.colormaps.get_cmap("Blues").resampled(len(boundaries) - 1)
norm        = BoundaryNorm(boundaries, cmap_obj.N, clip=True)

fig, ax = plt.subplots(figsize=(7, 5), dpi=120)
im = ax.pcolormesh(thresh_plot.lon, thresh_plot.lat, thresh_plot.values,
                   cmap=cmap_obj, norm=norm, shading="auto",
                   linewidth=0, edgecolors="none")
for lons_b, lats_b in get_india_outline(shapefile_path=shp_path):
    ax.plot(lons_b, lats_b, color="black", linewidth=0.8)
ax.patch.set_visible(False)

cbar = fig.colorbar(im, ax=ax, orientation="vertical", pad=0.02, shrink=0.85,
                    ticks=boundaries, spacing="uniform")
cbar.set_label("Threshold (mm / 5-day window)", fontsize=9)
cbar.ax.tick_params(labelsize=7)
cbar.set_ticklabels([str(int(b)) if b % 10 == 0 else "" for b in boundaries])

ax.set_title("Wet-spell Rainfall Threshold (MWmean) — IMD 1°", fontsize=10, fontweight="bold")
ax.set_xlabel("Longitude (°E)", fontsize=9)
ax.set_ylabel("Latitude (°N)", fontsize=9)
ax.tick_params(labelsize=8)
for side in ["top", "right", "bottom", "left"]:
    ax.spines[side].set_visible(True)
    ax.spines[side].set_linewidth(0.6)
plt.tight_layout()
plt.show()


## 🌧️ Step 5 — Load IMD Rainfall Data for 2025


In [ ]:
rainfall_da = load_rainfall(YEAR, dataset_cfg)

print(f"Rainfall shape : {rainfall_da.shape}")
print(f"Rainfall dims  : {rainfall_da.dims}")
print(f"Time range     : {str(rainfall_da.time.values[0])[:10]}  →  {str(rainfall_da.time.values[-1])[:10]}")
print(f"Lat range      : {float(rainfall_da.lat.min()):.1f} – {float(rainfall_da.lat.max()):.1f}")
print(f"Lon range      : {float(rainfall_da.lon.min()):.1f} – {float(rainfall_da.lon.max()):.1f}")
rainfall_da


## 📅 Step 6 — Detect Monsoon Onset Dates for 2025

Run the wet-spell / dry-spell veto algorithm grid-point by grid-point. Returns a `[lat, lon]` DataArray of `datetime64` onset dates (`NaT` where no valid onset is found).


In [ ]:
onset_da = detect_onset(
    rainfall_da=rainfall_da,
    threshold_da=thresh_da,
    year=YEAR,
    cfg=onset_cfg,
)

onset_doy = xr.where(
    onset_da.notnull(),
    onset_da.dt.dayofyear.astype(float),
    np.nan,
)
onset_doy.attrs["long_name"] = f"Monsoon Onset DOY {YEAR}"

valid_pts = int(onset_doy.notnull().sum())
total_pts = int(onset_doy.size)
print(f"Valid onset grid points : {valid_pts} / {total_pts}")
print(f"Earliest onset : {doy_to_date_string(float(onset_doy.min()))}")
print(f"Latest onset   : {doy_to_date_string(float(onset_doy.max()))}")
onset_doy


## 🗺️ Step 7 — Plot Onset Date Map (Monthly Stratified Colormap)

Each month (April–August) gets its own colour ramp so the progression of the monsoon is easy to read.


In [ ]:
from matplotlib.gridspec import GridSpec

SMALL_SIZE = 7; MEDIUM_SIZE = 9
label_fontsize = 7; tick_length = 3; tick_width = 0.6
panel_lw = 0.6; map_lw = 0.8

lon = onset_da.lon.values
lat = onset_da.lat.values

doy_vals = np.where(
    ~np.isnat(onset_da.values.astype("datetime64[ns]")),
    onset_da.dt.dayofyear.values.astype(float),
    np.nan,
)
if doy_vals.shape[0] == len(lon) and doy_vals.shape[1] == len(lat):
    doy_vals = doy_vals.T

# ── Monthly stratified colormap ───────────────────────────────────────────────
month_cmaps = {"Apr": plt.cm.cool, "May": plt.cm.YlOrBr,
               "Jun": plt.cm.Greens, "Jul": plt.cm.YlGnBu, "Aug": plt.cm.Purples}
month_doys  = {"Apr": (91, 121), "May": (121, 152), "Jun": (152, 182),
               "Jul": (182, 213), "Aug": (213, 244)}
N = 15
colors, bounds = [], []
for month, cmap in month_cmaps.items():
    d0, d1 = month_doys[month]
    colors.extend(cmap(np.linspace(0.3, 0.95, N)))
    bounds.extend(np.linspace(d0, d1, N, endpoint=False))
bounds.append(244)
cmap_m = ListedColormap(colors)
norm_m = mcolors.BoundaryNorm(bounds, cmap_m.N)

# ── pcolormesh edges ──────────────────────────────────────────────────────────
dlon = lon[1] - lon[0]; dlat = lat[1] - lat[0]
lon_edges = np.concatenate([lon - dlon/2, [lon[-1] + dlon/2]])
lat_edges = np.concatenate([lat - dlat/2, [lat[-1] + dlat/2]])
LON_e, LAT_e = np.meshgrid(lon_edges, lat_edges)
masked_data = np.ma.masked_invalid(doy_vals)

fig = plt.figure(figsize=(6, 6), dpi=150)
gs  = GridSpec(1, 14, figure=fig, hspace=0.1, wspace=0.3,
               left=0.08, right=0.92, top=0.85, bottom=0.15)
ax  = fig.add_subplot(gs[0, 0:12])

im = ax.pcolormesh(LON_e, LAT_e, masked_data, cmap=cmap_m, norm=norm_m,
                   shading="flat", linewidth=0, edgecolors="none")
for lons_b, lats_b in get_india_outline(shapefile_path=shp_path):
    ax.plot(lons_b, lats_b, color="black", linewidth=map_lw)

ax.patch.set_visible(False)
ax.set_xlim([lon[0] - 2, 100]); ax.set_ylim([lat[0] - 2, lat[-1]])
yticks = np.arange(lat[0] - 2, lat[-1] + 3, 8)
ax.set_yticks(yticks); ax.set_yticklabels([f"{int(y)}°N" for y in yticks], fontsize=label_fontsize)
xticks = np.arange(lon[0] - 2, lon[-1] + 3, 8)
ax.set_xticks(xticks); ax.set_xticklabels([f"{int(x)}°E" for x in xticks], fontsize=label_fontsize)
ax.set_title(f"IMD 1° Monsoon Onset — {YEAR}", fontsize=MEDIUM_SIZE, fontweight="bold")
ax.tick_params("both", length=tick_length, width=tick_width, which="major")
for side in ["top", "right", "bottom", "left"]:
    ax.spines[side].set_visible(True); ax.spines[side].set_linewidth(panel_lw)
ax.set_aspect("equal", adjustable="box")

pos = ax.get_position()
cb_h = pos.height * 0.9
cax  = fig.add_axes([pos.x1 + 0.02, pos.y0 + (pos.height - cb_h)/2, 0.022, cb_h])
cbar = fig.colorbar(im, cax=cax, orientation="vertical", spacing="proportional")
cbar.ax.minorticks_off()
ref = pd.Timestamp(f"{YEAR}-01-01")
tick_doys, tick_labels = [], []
for d0, d1 in month_doys.values():
    doy = d0
    while doy < d1:
        date = ref + pd.Timedelta(days=int(doy) - 1)
        tick_doys.append(doy); tick_labels.append(date.strftime("%d-%m")); doy += 7
cbar.set_ticks(tick_doys); cbar.set_ticklabels(tick_labels, fontsize=SMALL_SIZE - 0.5)
cbar.ax.tick_params(labelsize=SMALL_SIZE - 0.5, length=3, width=0.8, pad=2)
for month, (d0, d1) in month_doys.items():
    cbar.ax.axhline(d0, color="white", linewidth=1.5, zorder=5)
    cbar.ax.text(-0.5, (d0 + d1)/2, month, transform=cbar.ax.get_yaxis_transform(),
                 fontsize=SMALL_SIZE, va="center", ha="right", color="0.25", fontweight="bold")

out = f"{REPO_DIR}/output/onset_imd_{YEAR}_monthly.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {out}")


## 🔷 Step 8 — CMZ Polygon Overlay & Onset Statistics

Overlay the Core Monsoon Zone (CMZ) boundary on the onset map and print onset statistics for CMZ grid points only.


In [ ]:
# ── CMZ polygon (Rajeevan et al. 2010) ───────────────────────────────────────
cmz_lon = np.array([74, 85, 85, 86, 86, 87, 87, 88, 88, 88, 85, 85, 82, 82, 79, 79, 78, 78, 69, 69, 74, 74])
cmz_lat = np.array([18, 18, 19, 19, 20, 20, 21, 21, 21, 24, 24, 25, 25, 26, 26, 27, 27, 28, 28, 21, 21, 18])
cmz_path = Path(np.column_stack([cmz_lon, cmz_lat]))

# ── CMZ mask ──────────────────────────────────────────────────────────────────
LON_c, LAT_c = np.meshgrid(onset_da.lon.values, onset_da.lat.values)
pts           = np.column_stack([LON_c.ravel(), LAT_c.ravel()])
cmz_mask_2d   = cmz_path.contains_points(pts).reshape(LON_c.shape)
cmz_mask_da   = xr.DataArray(cmz_mask_2d, dims=["lat", "lon"],
                               coords={"lat": onset_da.lat, "lon": onset_da.lon})
onset_cmz     = onset_da.where(cmz_mask_da)
onset_doy_cmz = xr.where(onset_cmz.notnull(), onset_cmz.dt.dayofyear.astype(float), np.nan)

valid_cmz = int(onset_doy_cmz.notnull().sum())
total_cmz = int(cmz_mask_2d.sum())
mean_doy  = float(onset_doy_cmz.mean())
print(f"CMZ grid points with valid onset : {valid_cmz} / {total_cmz}")
print(f"CMZ earliest onset : {doy_to_date_string(float(onset_doy_cmz.min()))}")
print(f"CMZ latest   onset : {doy_to_date_string(float(onset_doy_cmz.max()))}")
print(f"CMZ mean     onset : {doy_to_date_string(mean_doy)}  (DOY {mean_doy:.1f})")

# ── Onset map with CMZ outline ────────────────────────────────────────────────
fig = plt.figure(figsize=(6, 6), dpi=150)
gs  = GridSpec(1, 14, figure=fig, hspace=0.1, wspace=0.3,
               left=0.08, right=0.92, top=0.85, bottom=0.15)
ax  = fig.add_subplot(gs[0, 0:12])

im = ax.pcolormesh(LON_e, LAT_e, masked_data, cmap=cmap_m, norm=norm_m,
                   shading="flat", linewidth=0, edgecolors="none")
for lons_b, lats_b in get_india_outline(shapefile_path=shp_path):
    ax.plot(lons_b, lats_b, color="black", linewidth=map_lw)
ax.plot(np.append(cmz_lon, cmz_lon[0]), np.append(cmz_lat, cmz_lat[0]),
        color="black", linewidth=2, linestyle="--", zorder=10, label="CMZ")
ax.legend(fontsize=SMALL_SIZE, loc="lower right", frameon=False)

ax.patch.set_visible(False)
ax.set_xlim([lon[0] - 2, 100]); ax.set_ylim([lat[0] - 2, lat[-1]])
ax.set_yticks(yticks); ax.set_yticklabels([f"{int(y)}°N" for y in yticks], fontsize=label_fontsize)
ax.set_xticks(xticks); ax.set_xticklabels([f"{int(x)}°E" for x in xticks], fontsize=label_fontsize)
ax.set_title(f"IMD 1° Monsoon Onset — {YEAR}  (CMZ outlined)", fontsize=MEDIUM_SIZE, fontweight="bold")
ax.tick_params("both", length=tick_length, width=tick_width, which="major")
for side in ["top", "right", "bottom", "left"]:
    ax.spines[side].set_visible(True); ax.spines[side].set_linewidth(panel_lw)
ax.set_aspect("equal", adjustable="box")

pos = ax.get_position()
cb_h = pos.height * 0.9
cax  = fig.add_axes([pos.x1 + 0.02, pos.y0 + (pos.height - cb_h)/2, 0.022, cb_h])
cbar = fig.colorbar(im, cax=cax, orientation="vertical", spacing="proportional")
cbar.ax.minorticks_off()
cbar.set_ticks(tick_doys); cbar.set_ticklabels(tick_labels, fontsize=SMALL_SIZE - 0.5)
cbar.ax.tick_params(labelsize=SMALL_SIZE - 0.5, length=3, width=0.8, pad=2)
for month, (d0, d1) in month_doys.items():
    cbar.ax.axhline(d0, color="white", linewidth=1.5, zorder=5)
    cbar.ax.text(-0.5, (d0 + d1)/2, month, transform=cbar.ax.get_yaxis_transform(),
                 fontsize=SMALL_SIZE, va="center", ha="right", color="0.25", fontweight="bold")

out = f"{REPO_DIR}/output/onset_imd_{YEAR}_cmz.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {out}")


## 📈 Step 9 — Grid-point Daily Rainfall Time-series

Plot the daily rainfall for a chosen grid cell with the local threshold and detected onset date overlaid.  
Change `SEL_LAT` / `SEL_LON` to any grid point you like.


In [ ]:
# ── Choose grid cell ──────────────────────────────────────────────────────────
SEL_LAT = 19.5
SEL_LON = 76.5

rain_cell   = rainfall_da.sel(lat=SEL_LAT, lon=SEL_LON, method="nearest")
thresh_cell = float(thresh_da.sel(lat=SEL_LAT, lon=SEL_LON, method="nearest").squeeze())
onset_cell  = onset_da.sel(lat=SEL_LAT, lon=SEL_LON, method="nearest")

actual_lat = float(rain_cell.lat)
actual_lon = float(rain_cell.lon)
onset_ts   = (pd.Timestamp(onset_cell.values)
              if not np.isnat(onset_cell.values.astype("datetime64[ns]")) else None)

t_start   = pd.Timestamp(f"{YEAR}-04-01")
t_end     = pd.Timestamp(f"{YEAR}-08-31")
rain_plot = rain_cell.to_series().loc[t_start:t_end]
times     = rain_plot.index

fig, ax = plt.subplots(figsize=(9, 3), dpi=150)
ax.plot(times, rain_plot.values, color="steelblue", linewidth=0.9,
        marker="o", markersize=4.5, markerfacecolor="steelblue", markeredgewidth=0,
        label="Daily rainfall (mm)", zorder=3)
ax.axhline(thresh_cell, color="darkgreen", linewidth=1.2, linestyle="--",
           label=f"Threshold: {thresh_cell:.1f} mm", zorder=2)
if onset_ts:
    ax.axvline(onset_ts, color="crimson", linewidth=1.5, zorder=4,
               label=f"Onset: {onset_ts.strftime('%d-%b-%Y')}")
    ax.text(onset_ts, ax.get_ylim()[1] * 0.97, f" {onset_ts.strftime('%d-%b')}",
            color="crimson", fontsize=7, va="top", ha="left")

ax.set_ylabel("Daily rainfall (mm)", fontsize=8)
ax.set_xlabel("Date", fontsize=8)
ax.set_title(f"Daily Rainfall — {YEAR}  |  {actual_lat:.1f}°N, {actual_lon:.1f}°E",
             fontsize=9, fontweight="bold")
ax.legend(fontsize=7, frameon=False, loc="upper right")
ax.tick_params(labelsize=7)
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter("%d-%b"))
ax.xaxis.set_major_locator(plt.matplotlib.dates.MonthLocator())
ax.xaxis.set_minor_locator(plt.matplotlib.dates.WeekdayLocator(byweekday=0))
ax.tick_params(axis="x", rotation=30)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
out = f"{REPO_DIR}/output/timeseries_{YEAR}_{actual_lat:.1f}N_{actual_lon:.1f}E.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {out}")


## ⏱️ Step 10 — First Wet Spell vs Onset: Dry-spell Veto Delay

Compute the **first wet spell date** (same criteria, no dry-spell veto) and compare it to the detected onset date. The difference shows how much the dry-spell veto delays the onset.


In [ ]:
first_wet_da = detect_first_wet_spell(rainfall_da, thresh_da, YEAR, onset_cfg)

def doy_arr(da):
    ref = np.datetime64(f"{YEAR}-01-01", "D")
    return np.where(pd.notna(da.values),
                    (da.values.astype("datetime64[D]") - ref).astype(float) + 1,
                    np.nan)

fws_doy = doy_arr(first_wet_da)
ons_doy = doy_arr(onset_da)
delay_vals = ons_doy - fws_doy

delay_da = xr.DataArray(
    delay_vals,
    coords=[("lat", onset_da.lat.values), ("lon", onset_da.lon.values)],
    name="onset_delay_days",
    attrs={"description": "Onset − first wet spell (days); positive = delayed by dry-spell veto"},
)

valid_delays = delay_da.values[np.isfinite(delay_da.values)]
print(f"Median delay : {np.median(valid_delays):.1f} days")
print(f"Max delay    : {np.nanmax(valid_delays):.0f} days")
print(f"Points with zero delay (no veto): {int((valid_delays == 0).sum())}")


In [ ]:
# ── Shared colormap for date panels ──────────────────────────────────────────
doy_bounds_c = list(range(91, 244, 8))
if doy_bounds_c[-1] < 243:
    doy_bounds_c.append(243)
N_bins     = len(doy_bounds_c) - 1
cmap_seq   = plt.cm.turbo
cmap_shared = mcolors.ListedColormap([cmap_seq(i / (N_bins - 1)) for i in range(N_bins)])
norm_shared = mcolors.BoundaryNorm(doy_bounds_c, N_bins)

# ── Delay colormap ────────────────────────────────────────────────────────────
max_delay    = 50; step = 4
max_bin      = int(np.ceil(max_delay / step)) * step
delay_bounds = list(range(0, max_bin + step, step))
N_delay      = len(delay_bounds) - 1
cmap_delay   = mcolors.ListedColormap(plt.cm.YlOrRd(np.linspace(0.05, 0.95, N_delay)))
norm_delay   = mcolors.BoundaryNorm(delay_bounds, N_delay)

# ── Figure ────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(12, 5), dpi=150)
gs  = mgridspec.GridSpec(1, 3, figure=fig, wspace=0.15,
                          left=0.06, right=0.97, top=0.88, bottom=0.18)
map_extent   = [65, 100, 6, 38]
panel_titles = ["First Wet Spell Date", "Onset Date",
                "Dry spell length: onset − first wet spell"]

for pi, (doy_data, da_obj) in enumerate([(fws_doy, first_wet_da), (ons_doy, onset_da)]):
    ax = fig.add_subplot(gs[pi])
    ax.set_aspect("equal", adjustable="box")
    im = ax.pcolormesh(da_obj.lon.values, da_obj.lat.values, doy_data,
                       cmap=cmap_shared, norm=norm_shared, shading="nearest", rasterized=True)
    for lons_b, lats_b in get_india_outline(shapefile_path=shp_path):
        ax.plot(lons_b, lats_b, color="black", linewidth=0.8)
    ax.patch.set_visible(False)
    ax.set_xlim(*map_extent[:2]); ax.set_ylim(*map_extent[2:])
    ax.set_title(panel_titles[pi], fontsize=10, fontweight="bold")
    ax.set_xlabel("Longitude (°E)", fontsize=8); ax.tick_params(labelsize=7)
    if pi == 0: ax.set_ylabel("Latitude (°N)", fontsize=8)
    else: ax.set_yticklabels([])
    cbar = fig.colorbar(im, ax=ax, orientation="horizontal", pad=0.1,
                        fraction=0.046, aspect=16,
                        boundaries=doy_bounds_c, ticks=doy_bounds_c[::3])
    tick_dates = [pd.Timestamp(f"{YEAR}-01-01") + pd.Timedelta(days=int(d) - 1)
                  for d in doy_bounds_c[::3]]
    cbar.ax.set_xticklabels([t.strftime("%d-%b") for t in tick_dates],
                             rotation=0, fontsize=6.5, ha="right")

ax2 = fig.add_subplot(gs[2])
ax2.set_aspect("equal", adjustable="box")
im2 = ax2.pcolormesh(delay_da.lon.values, delay_da.lat.values, delay_da.values,
                      cmap=cmap_delay, norm=norm_delay, shading="nearest", rasterized=True)
for lons_b, lats_b in get_india_outline(shapefile_path=shp_path):
    ax2.plot(lons_b, lats_b, color="black", linewidth=0.8)
ax2.patch.set_visible(False)
ax2.set_xlim(*map_extent[:2]); ax2.set_ylim(*map_extent[2:])
ax2.set_title(panel_titles[2], fontsize=10, fontweight="bold")
ax2.set_xlabel("Longitude (°E)", fontsize=8); ax2.set_yticklabels([]); ax2.tick_params(labelsize=7)
cbar2 = fig.colorbar(im2, ax=ax2, orientation="horizontal", pad=0.1,
                      fraction=0.046, aspect=16,
                      boundaries=delay_bounds, ticks=delay_bounds, spacing="uniform")
cbar2.ax.set_xticklabels([str(b) for b in delay_bounds], fontsize=7)
cbar2.ax.tick_params(labelsize=7)

out = f"{REPO_DIR}/output/comparison_fws_vs_onset_{YEAR}.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {out}")
